In [0]:
# %pip install kafka-python

In [0]:
# dbutils.library.restartPython()

In [0]:
%run ../utils/adls_auth

In [0]:
from pyspark.sql.functions import from_json, col, current_timestamp
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

EVENT_HUB_NAMESPACE = "ehns-nyc-taxi-dev"
EVENT_HUB_TOPIC = "trips-stream"
CHECKPOINT_PATH = "abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/_checkpoints/trips_stream"
OUTPUT_PATH = "abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/trips_stream_raw"

eh_listen_conn_str = dbutils.secrets.get(scope="kv-nyc-taxi-scope", key="eh-listen-connection-string")


In [0]:

clean_conn_str = eh_listen_conn_str.split(";EntityPath=")[0] if ";EntityPath=" in eh_listen_conn_str else eh_listen_conn_str

kafka_options = {
    "kafka.bootstrap.servers": f"{EVENT_HUB_NAMESPACE}.servicebus.windows.net:9093",
    "subscribe": EVENT_HUB_TOPIC,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": (
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="$ConnectionString" password="{clean_conn_str}";'
    ),
    "startingOffsets": "earliest",
    "failOnDataLoss": "false"
}

event_schema = StructType([
    StructField("vendor_id", IntegerType()),
    StructField("pickup_location_id", IntegerType()),
    StructField("dropoff_location_id", IntegerType()),
    StructField("trip_distance", DoubleType()),
    StructField("fare_amount", DoubleType()),
    StructField("passenger_count", IntegerType()),
    StructField("event_time", StringType()),
    StructField("source", StringType()),
])

raw_stream_df = spark.readStream.format("kafka").options(**kafka_options).load()


In [0]:

parsed_df = (
    raw_stream_df
    .selectExpr("CAST(value AS STRING) as json_value", "timestamp as kafka_timestamp")
    .select(from_json(col("json_value"), event_schema).alias("data"), "kafka_timestamp")
    .select("data.*", "kafka_timestamp")
    .withColumn("_consumed_at", current_timestamp())
)

query = (
    parsed_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime="2 minutes")
    .start(OUTPUT_PATH)
)

print(f"Streaming query started: {query.id}")
print("Run query.stop() in a separate cell when done or let this notebook's job run time out.")

In [0]:
# query.stop()